In [1]:
import pandas as pd

from ssfaitk.models import EffortClassifier

# ---------------------------------------------------------------------
# 1. Load the data
# ---------------------------------------------------------------------
df = pd.read_csv("../../data/dfKenyaAll_Tracks_df_cleaned.csv")

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.head()


Columns: ['Observer', 'Date', 'ident', 'Latitude', 'Longitude', 'y_proj', 'x_proj', 'new_trk', 'new_seg', 'altitude', 'model', 'ltime', 'time_EAT', 'Location', 'Activity', 'Trip_Haul_ID', 'Trip_ID']
Shape: (193934, 17)


/var/folders/kj/2hjk7hzn2qz7f8rsq4mzqnm40000gp/T/ipykernel_61715/3066303021.py:7: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/dfKenyaAll_Tracks_df_cleaned.csv")


,Observer,Date,ident,Latitude,Longitude,y_proj,x_proj,new_trk,new_seg,altitude,model,ltime,time_EAT,Location,Activity,Trip_Haul_ID,Trip_ID
0,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.526989,39.503832,-4.526989,39.503832,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:48:00,2016-01-06 08:48:00,Gazi,Fishing,62_2,62
1,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.527075,39.503274,-4.527075,39.503274,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:50:00,2016-01-06 08:50:00,Gazi,Fishing,62_2,62
2,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.524565,39.504626,-4.524565,39.504626,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:52:00,2016-01-06 08:52:00,Gazi,Fishing,62_2,62
3,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.524114,39.504347,-4.524114,39.504347,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:54:00,2016-01-06 08:54:00,Gazi,Fishing,62_2,62
4,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.523449,39.505055,-4.523449,39.505055,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:56:00,2016-01-06 08:56:00,Gazi,Fishing,62_2,62


In [2]:
df['Activity'].unique()

array(['Fishing', 'Sailing', 'Searching'], dtype=object)

In [7]:
# 1️⃣ Load pre-trained model (RF)
clf = EffortClassifier.load_trained("rf")

# 2️⃣ Predict directly on new data (can include many trips)
preds = clf.predict_df(df)


preds.head()

/Users/altarturhamza/Library/CloudStorage/OneDrive-CGIAR/GitHub/ssf-ai-toolkit/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/altarturhamza/Library/CloudStorage/OneDrive-CGIAR/GitHub/ssf-ai-toolkit/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[INFO] 2025-11-03 15:24:38,424 ssfaitk.models.effo

,Observer,Date,ident,Latitude,Longitude,y_proj,x_proj,new_trk,new_seg,altitude,model,ltime,time_EAT,Location,Activity,Trip_Haul_ID,Trip_ID,effort_pred,effort_prob
0,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.526989,39.503832,-4.526989,39.503832,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:48:00,2016-01-06 08:48:00,Gazi,Fishing,62_2,62,0,0.0700
1,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.527075,39.503274,-4.527075,39.503274,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:50:00,2016-01-06 08:50:00,Gazi,Fishing,62_2,62,0,0.0525
2,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.524565,39.504626,-4.524565,39.504626,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:52:00,2016-01-06 08:52:00,Gazi,Fishing,62_2,62,0,0.0425
3,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.524114,39.504347,-4.524114,39.504347,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:54:00,2016-01-06 08:54:00,Gazi,Fishing,62_2,62,0,0.0375
4,Abudhabi Jambia,2016-01-06,65_H_2_F,-4.523449,39.505055,-4.523449,39.505055,NaN,NaN,NaN,legacy 2015_16 data,2016/01/06 08:56:00,2016-01-06 08:56:00,Gazi,Fishing,62_2,62,0,0.0275


In [27]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

from ssfaitk.eval.metrics import accuracy_score

df['Activity_bin'] = df['Activity'].replace({
    'Fishing':'fishing_search', 'Searching':'fishing_search', 'Sailing':'sailing_travel', 'Traveling':'sailing_travel'
}).fillna('sailing_travel')
y_true = df['Activity_bin']
le1 = joblib.load("../../src/ssfaitk/models/effort/artifacts/label_encoder_model.joblib")
y1_te_enc = le1.transform(y_true)


y_pred = preds["effort_pred"].values
acc = accuracy_score(y1_te_enc, y_pred)
f1w = f1_score(y1_te_enc, y_pred, average="weighted")

# stage1_results.append((name, acc, f1w))
print("\n[Stage-1]\n", classification_report(y1_te_enc, y_pred, target_names=le1.classes_))



[Stage-1]
                 precision    recall  f1-score   support

fishing_search       0.39      0.40      0.40     79461
sailing_travel       0.58      0.57      0.57    114473

      accuracy                           0.50    193934
     macro avg       0.49      0.49      0.49    193934
  weighted avg       0.50      0.50      0.50    193934



/Users/altarturhamza/Library/CloudStorage/OneDrive-CGIAR/GitHub/ssf-ai-toolkit/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
